RESET TIMESTAMP

In [1]:
import os
from datetime import datetime, timedelta, timezone
import shutil
from pathlib import Path
import time

In [ ]:
# Configurazione
SOURCE_DIR = Path("AudioSamplesOldTimestamp")
DEST_DIR = Path("AudioSamplesNewTimestamp")
OLD_DATE_FORMAT = "%Y/%m/%d,%H:%M:%S"
NEW_DATE_FORMAT = "%Y-%m-%dT%H:%M:%S%z"

if DEST_DIR.exists():
    shutil.rmtree(DEST_DIR)  # Rimuove la cartella e tutto il contenuto

DEST_DIR.mkdir(parents=True) # La ricrea vuota

In [3]:
def process_all_files():
    # Crea la cartella di destinazione se non esiste
    if not os.path.exists(DEST_DIR):
        os.makedirs(DEST_DIR)
        print(f"Cartella '{DEST_DIR}' creata.")

    for filename in os.listdir(SOURCE_DIR):
        if filename.lower().endswith(".wav"):
            src_path = os.path.join(SOURCE_DIR, filename)
            dest_path = os.path.join(DEST_DIR, filename)
            
            print(f"Elaborazione: {filename}...")
            success = rewrite_wav_with_new_metadata(src_path, dest_path)
            if success:
                print(f"  -> Salvato in {DEST_DIR}")

def rewrite_wav_with_new_metadata(src_path, dest_path):
    with open(src_path, "rb") as f:
        content = bytearray(f.read())

    # 1. Trova il tag "tmst"
    tag_pos = content.find(b"tmst")
    if tag_pos == -1:
        print(f"  [Errore] Tag 'tmst' non trovato.")
        return False

    # 2. Leggi lunghezza vecchia e valore vecchio
    old_len = int.from_bytes(content[tag_pos+4:tag_pos+8], "little")
    old_val_start = tag_pos + 8
    old_val_end = old_val_start + old_len
    old_val_str = content[old_val_start:old_val_end].rstrip(b"\x00").decode("ascii")

    try:
        # 3. Calcola il nuovo timestamp
        dt = datetime.strptime(old_val_str, OLD_DATE_FORMAT)
        dt_corrected = dt + timedelta(hours=1)
        tz_italy_solar = timezone(timedelta(hours=1))
        dt_corrected = dt_corrected.replace(tzinfo=tz_italy_solar)
        new_val_str = dt_corrected.strftime(NEW_DATE_FORMAT)
        new_val_bytes = new_val_str.encode("ascii")
        new_len = len(new_val_bytes)
        print(f"  Vecchio timestamp: {old_val_str} -> Nuovo timestamp: {new_val_str}")

        # 4. Aggiorna il contenuto (Sostituzione con cambio dimensione)
        # Sostituiamo i vecchi byte con i nuovi
        content[old_val_start:old_val_end] = new_val_bytes
        
        # 5. Aggiorna la dimensione del chunk del metadato (4 byte dopo 'tmst')
        content[tag_pos+4:tag_pos+8] = new_len.to_bytes(4, "little")

        # 6. Aggiorna la dimensione del chunk "LIST" (INFO)
        # Dobbiamo trovare il chunk LIST che contiene questo tmst
        # (Semplificazione: cerchiamo all'indietro il tag LIST più vicino)
        list_pos = content.rfind(b"LIST", 0, tag_pos)
        if list_pos != -1:
            current_list_size = int.from_bytes(content[list_pos+4:list_pos+8], "little")
            new_list_size = current_list_size + (new_len - old_len)
            content[list_pos+4:list_pos+8] = new_list_size.to_bytes(4, "little")

        # 7. Aggiorna la dimensione totale del file RIFF (byte 4-8 del file)
        current_riff_size = int.from_bytes(content[4:8], "little")
        new_riff_size = current_riff_size + (new_len - old_len)
        content[4:8] = new_riff_size.to_bytes(4, "little")

        # 8. Scrittura del nuovo file
        with open(dest_path, "wb") as f_out:
            f_out.write(content)
        return True

    except Exception as e:
        print(f"  [Errore] Fallito parsing data '{old_val_str}': {e}")
        return False

In [4]:
process_all_files()

Elaborazione: audio_0001b365ea502220f6a020a344147186b53f26c528a8b0b7e0545690a116a00c.wav...
  Vecchio timestamp: 2026/02/24,08:21:34 -> Nuovo timestamp: 2026-02-24T09:21:34+0100
  -> Salvato in AudioSamplesNewTimestamp
Elaborazione: audio_0029f00646b5b48dd4a465e27580a46809b344e3e53568469cd92ed445f24158.wav...
  Vecchio timestamp: 2026/03/12,01:34:15 -> Nuovo timestamp: 2026-03-12T02:34:15+0100
  -> Salvato in AudioSamplesNewTimestamp
Elaborazione: audio_002ec2062c34fed7eef96945da6f60a6c37678aff80dadf6ac00f29c90fc4827.wav...
  Vecchio timestamp: 2026/03/09,16:53:20 -> Nuovo timestamp: 2026-03-09T17:53:20+0100
  -> Salvato in AudioSamplesNewTimestamp
Elaborazione: audio_00865d39aacb496372a9338bce90985aed4d1fe0998c328d68cdff4151bcee6b.wav...
  Vecchio timestamp: 2026/03/12,00:43:12 -> Nuovo timestamp: 2026-03-12T01:43:12+0100
  -> Salvato in AudioSamplesNewTimestamp
Elaborazione: audio_0090959c000a0f8497526f0161fe259fbdb0947d3f422d1a4e5464f1fa01c491.wav...
  Vecchio timestamp: 2026/03/14,